# 10 · Объединённая цель: учитель-промпт, где он прав, иначе центроид верных состояний

**Зачем** (JOURNAL, прогон 09). Без градиента и без LoRA две цели аффинной поправки в точке вызова
дополняют друг друга: `linmap_prompt` (учитель — та же модель с промптом explicit_state) переносится на
новые типы требований (0.83 против 0.88 у LoRA), а `linmap` (центроид верных состояний базы) точнее на
знакомых (0.76 против 0.70). `linmap_mix` берёт цель учителя там, где он решил верно, иначе — цель `linmap`.

Те же 110 промптов train; выбор на val; сетка силы до 4.0 (у `linmap_prompt` 3.0 была на краю).
Проверка — на **свежем** наборе переноса (seed 404): набор 303 повлиял на выбор этого метода.
Результат **разведочный**.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [ ]:
cfg = ExperimentConfig.make(
    name="mix-target-qwen3-1.7b", hypothesis="explore", notebook="notebooks/10_mix_target.ipynb",
    notes="Объединённая цель linmap_mix: учитель-промпт, где он прав, иначе центроид верных состояний.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "v3",
        "eval.methods": ["baseline", "lora", "explicit_state", "linmap", "linmap_prompt", "linmap_mix"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
        "act.ranks": [8, 16],
        "act.alphas": [1.0, 1.5, 2.0, 3.0, 4.0],
        "episode.on_val": False,
        "transfer.seed": 404,
        "transfer.episode_methods": ["baseline", "lora", "linmap", "linmap_prompt", "linmap_mix"],
    },
)
exp = Experiment(cfg)
exp.run("fit")
exp.results_table("val")

### Подбор на val

In [ ]:
f = exp.state["fitted"]
t = f.get("linmap_mix", {})
print(f"учитель {t.get('teacher')} на train: успех {t.get('teacher_train_success', float('nan')):.3f} "
      f"(база {t.get('base_train_success', float('nan')):.3f})")
for m in ("linmap_mix", "linmap_prompt", "linmap"):
    t = exp.fit_table(m)
    if t.empty:
        print(m, f[m].get("reason")); continue
    display(Markdown(f"**{m}** — лучшие конфигурации на val"))
    cols = ["layer", "rank", "alpha", "train_r2", "success", "sensitivity", "specificity", "spec_drop", "fixed", "broken"]
    display(t.sort_values("success", ascending=False)[cols].head(8).round(3))

### Свежий набор переноса (seed 404): знакомые и новые типы требований

In [ ]:
exp.transfer()
for g in ("seen", "unseen"):
    display(Markdown(f"**{g}**"))
    display(exp.results_table(f"transfer:{g}"))

### Цена: подготовка и вывод

In [ ]:
d = exp.backend.hidden_size
cost = []
lr = f["lora"]["row"]
cost.append({"method": "lora", "prep": "градиент по весам (копия fp32)", "prep_seconds": lr["train_seconds"],
             "params": lr["n_trainable"]})
for m, how in (("linmap", "закрытая форма"), ("linmap_prompt", "закрытая форма + учитель-промпт (генерация)"),
               ("linmap_mix", "закрытая форма + учитель-промпт (генерация)")):
    r, c = f[m]["row"], f[m].get("cost", {})
    if r:
        cost.append({"method": m, "prep": how, "prep_seconds": c.get("acts_seconds", 0) + c.get("fit_seconds", 0),
                     "params": 2 * d * r["rank"] + d})
cost = pd.DataFrame(cost)
inf = exp.results_table("transfer:seen").set_index("method")[["tokens", "calls", "p50_ms"]]
cost.merge(inf, left_on="method", right_index=True, how="outer").reset_index(drop=True)

## Вывод → журнал

In [ ]:
display(Markdown(exp.report(
    conclusion="""…""",
    next_step="""…""",
)))